# Text-to-Speech Generation with HuggingFace Transformers

Generate high-quality speech from text using two modern TTS models:

| Model | Size | Highlights |
|---|---|---|
| `suno/bark-small` | ~600MB | Expressive, multi-voice, supports laughter/music/non-verbal sounds |
| `microsoft/speecht5_tts` | ~400MB | Fast, lightweight, controllable via speaker embeddings |

> **GPU recommended** for Bark. SpeechT5 runs well on CPU.

## Installation

In [1]:
!pip install -q transformers datasets accelerate scipy

## Imports & Device Setup

In [2]:
import torch
import numpy as np
import scipy.io.wavfile as wavfile
from IPython.display import Audio, display
from transformers import (
    pipeline,
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device.upper()}")

Using: CUDA


## Helper — Play and Save Audio

In [3]:
def play_audio(audio_array, sample_rate, filename=None):
    """Display audio in the notebook and optionally save it as a WAV file."""
    # Ensure 1-D for playback
    audio = np.array(audio_array).squeeze()
    display(Audio(audio, rate=sample_rate))

    if filename:
        # Normalise to int16 range for WAV export
        audio_int16 = (audio / np.max(np.abs(audio)) * 32767).astype(np.int16)
        wavfile.write(filename, sample_rate, audio_int16)
        print(f"Saved: {filename}")

---
## Part 1 — Bark (Expressive TTS)

Bark is a transformer-based TTS model that generates highly expressive speech.  
It supports **voice presets**, non-verbal sounds like `[laughs]`, `[sighs]`, music, and multiple languages.

In [4]:
bark_pipe = pipeline(
    "text-to-speech",
    model="suno/bark-small",
    device=device,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

speaker_embeddings_path.json: 0.00B [00:00, ?B/s]

### Basic Generation

In [5]:
text = "Python is a high-level, general-purpose programming language."

output = bark_pipe(text, forward_params={"do_sample": True})

print(f"Sample rate: {output['sampling_rate']} Hz")
print(f"Audio shape: {np.array(output['audio']).shape}")

play_audio(output["audio"], output["sampling_rate"], filename="bark_basic.wav")

Passing `generation_config` together with generation-related arguments=({'min_eos_p', 'do_sample', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text

Sample rate: 24000 Hz
Audio shape: (176000,)


Saved: bark_basic.wav


### Voice Presets

Bark ships with preset speaker voices (`v2/en_speaker_0` through `v2/en_speaker_9`).  
Non-English presets also available: `v2/de_speaker_0`, `v2/fr_speaker_0`, etc.

In [ ]:
voice_presets = ["v2/en_speaker_1", "v2/en_speaker_6", "v2/en_speaker_9"]
voice_text = "Hello! I am demonstrating different voice presets available in Bark."
        
for preset in voice_presets:
    print(f"\n--- Voice: {preset} ---")
    out = bark_pipe(
        voice_text,
        forward_params={"do_sample": True},
        # generate_kwargs={"voice_preset": preset},
    )
    play_audio(out["audio"], out["sampling_rate"])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Voice: v2/en_speaker_1 ---


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Voice: v2/en_speaker_6 ---


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Voice: v2/en_speaker_9 ---


Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

### Non-Verbal Sounds and Expressive Tags

Bark understands special tags embedded in text: `[laughs]`, `[sighs]`, `[gasps]`, `[clears throat]`, `...` for pauses.

In [ ]:
expressive_texts = [
    "Wait, are you serious? [laughs] That's the funniest thing I've heard all week!",
    "I'm not sure about this... [sighs] Let me think it over.",
    "The results are in... and they are... absolutely incredible!",
]

for text in expressive_texts:
    print(f"\nText: {text!r}")
    out = bark_pipe(text, forward_params={"do_sample": True})
    play_audio(out["audio"], out["sampling_rate"])

### Multilingual Generation

In [ ]:
multilingual_examples = [
    ("v2/en_speaker_6", "Machine learning is transforming the world."),
    ("v2/es_speaker_0", "El aprendizaje automático está transformando el mundo."),
    ("v2/fr_speaker_0", "L'apprentissage automatique transforme le monde."),
]

for preset, text in multilingual_examples:
    print(f"\nPreset: {preset}")
    print(f"Text:   {text}")
    out = bark_pipe(
        text,
        forward_params={"do_sample": True},
        generate_kwargs={"voice_preset": preset},
    )
    play_audio(out["audio"], out["sampling_rate"])

---
## Part 2 — SpeechT5 (Fast, Lightweight TTS)

SpeechT5 is Microsoft's encoder-decoder TTS model.  
It uses **speaker embeddings** (x-vectors) extracted from real speakers to control voice characteristics — more deterministic than Bark.

In [ ]:
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model  = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder    = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# Load a dataset of pre-computed speaker x-vectors (CMU ARCTIC)
xvector_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")

SAMPLE_RATE = 16000  # SpeechT5 always outputs at 16 kHz

print(f"Speaker embeddings dataset size: {len(xvector_dataset)}")

In [ ]:
def speecht5_generate(text, speaker_idx=7306):
    """Generate speech with SpeechT5 using a speaker embedding by index."""
    inputs = processor(text=text, return_tensors="pt").to(device)
    speaker_embedding = (
        torch.tensor(xvector_dataset[speaker_idx]["xvector"])
        .unsqueeze(0)
        .to(device)
    )
    with torch.no_grad():
        speech = tts_model.generate_speech(
            inputs["input_ids"], speaker_embedding, vocoder=vocoder
        )
    return speech.cpu().numpy()

### Basic Generation

In [ ]:
text = "Python is a high-level, general-purpose programming language."

audio = speecht5_generate(text)
play_audio(audio, SAMPLE_RATE, filename="speecht5_basic.wav")

### Different Speakers

Different `speaker_idx` values correspond to different speakers in the CMU Arctic dataset.

In [ ]:
speaker_text = "The future of artificial intelligence is both exciting and full of challenges."

for idx in [7306, 2500, 5000]:
    print(f"\n--- Speaker index: {idx} ---")
    audio = speecht5_generate(speaker_text, speaker_idx=idx)
    play_audio(audio, SAMPLE_RATE)

### Long-Form Text Generation

SpeechT5 has a 600-token input limit. For longer texts, split into sentences and concatenate.

In [ ]:
long_text = """
Machine learning is a subset of artificial intelligence.
It enables systems to learn and improve from experience without being explicitly programmed.
Neural networks are inspired by the structure of the human brain.
"""

sentences = [s.strip() for s in long_text.strip().split("\n") if s.strip()]
audio_segments = [speecht5_generate(sent) for sent in sentences]

# Add a short silence (0.3s) between sentences
silence = np.zeros(int(0.3 * SAMPLE_RATE))
combined = np.concatenate([np.concatenate([seg, silence]) for seg in audio_segments])

play_audio(combined, SAMPLE_RATE, filename="speecht5_long.wav")

---
## Summary — Model Comparison

| Feature | Bark Small | SpeechT5 |
|---|---|---|
| Speed | Slower | Fast |
| Expressiveness | High (laughs, sighs, music) | Moderate |
| Voice control | Presets (`v2/en_speaker_X`) | Speaker embeddings |
| Multilingual | Yes | English only |
| Long text | Native | Split manually |
| VRAM (fp16) | ~4GB | ~1GB |
| Output sample rate | 24 kHz | 16 kHz |